# Dataset Preparation


## Imports


In [1]:
import os
import subprocess
import xml.etree.ElementTree as ET

import numpy as np

## Constants


In [2]:
np.random.seed(42)

os.environ["SUMO_HOME"] = os.path.join(os.environ["CONDA_PREFIX"], "Lib", "site-packages", "sumo")
SUMO_HOME = os.environ["SUMO_HOME"]
os.environ["PATH"] += os.pathsep + os.path.join(SUMO_HOME, "bin")
os.environ["PATH"] += os.pathsep + os.path.join(SUMO_HOME, "tools")

DATASET_DIR = os.path.join("athens-osmWebWizard")

NETWORK_PATH = os.path.join(DATASET_DIR, "osm.net.xml.gz")
RANDOM_TRIPS_PATH = os.path.join(SUMO_HOME, "tools", "randomTrips.py")
DUAROUTER_PATH = os.path.join(SUMO_HOME, "bin", "duarouter.exe")

VEHICLE_TYPES_PROBABILITIES = {"car": 0.70, "ldv": 0.10, "truck": 0.05, "bus": 0.15}

TRAFFIC_VOLUMES = [1, 2, 6, 5, 3, 1, 2, 6, 4, 3]

## Traffic Volumes


In [3]:
processed_traffic_volumes = [12 / x for x in TRAFFIC_VOLUMES]

train_traffic_volumes = list(np.array(processed_traffic_volumes))
test_traffic_volumes = [v * np.random.uniform(0.95, 1.05) for v in train_traffic_volumes]

## Random Trips and Routes


In [4]:
def generate_random_trips_and_routes(traffic_volumes, prefix, simulation_start=0, simulation_end=36000):
    traffic_volumes_str = ",".join(str(v) for v in traffic_volumes)
    trips_file = os.path.join(DATASET_DIR, f"{prefix}-trips.rou.xml")
    routes_file = os.path.join(DATASET_DIR, f"{prefix}-routes.rou.xml")

    command = [
        "python",
        RANDOM_TRIPS_PATH,
        "--net-file",
        NETWORK_PATH,
        "--begin",
        str(simulation_start),
        "--end",
        str(simulation_end),
        "--period",
        traffic_volumes_str,
        "-o",
        trips_file,
        "-r",
        routes_file,
        "--edge-param",
        "main",
    ]

    print("Executing:", " ".join(command))
    result = subprocess.run(command, capture_output=True, text=True)
    if result.stderr:
        print("Errors/Warnings from randomTrips:")
        print(result.stderr)

    if os.path.exists(trips_file) and os.path.exists(routes_file):
        os.remove(trips_file)
        print("Success:", routes_file)
    else:
        print("Failed:", trips_file)


print("Generating train random trips and routes...")
generate_random_trips_and_routes(train_traffic_volumes, "train")
print("Generating test random trips and routes...")
generate_random_trips_and_routes(test_traffic_volumes, "test")

Generating train random trips and routes...
Executing: python C:\Users\george\miniconda3\envs\thesis\Lib\site-packages\sumo\tools\randomTrips.py --net-file athens-osmWebWizard\osm.net.xml.gz --begin 0 --end 36000 --period 12.0,6.0,2.0,2.4,4.0,12.0,6.0,2.0,3.0,4.0 -o athens-osmWebWizard\train-trips.rou.xml -r athens-osmWebWizard\train-routes.rou.xml --edge-param main
Success: athens-osmWebWizard\train-routes.rou.xml
Generating test random trips and routes...
Executing: python C:\Users\george\miniconda3\envs\thesis\Lib\site-packages\sumo\tools\randomTrips.py --net-file athens-osmWebWizard\osm.net.xml.gz --begin 0 --end 36000 --period 11.849448142616835,6.27042858384595,2.046398788362281,2.4236780362072885,3.8624074561769746,11.587193424403441,5.73485016730092,2.073235229154987,3.0303345035229627,4.083229031118418 -o athens-osmWebWizard\test-trips.rou.xml -r athens-osmWebWizard\test-routes.rou.xml --edge-param main
Success: athens-osmWebWizard\test-routes.rou.xml


## Vehicle IDs


In [5]:
def update_vehicle_ids(prefix):
    routes_file = os.path.join(DATASET_DIR, f"{prefix}-routes.rou.xml")
    if not os.path.exists(routes_file):
        print(f"Routes file not found: {routes_file}")
        return

    tree = ET.parse(routes_file)
    root = tree.getroot()

    vehicle_id = 0
    for vehicle in root.findall("vehicle"):
        vehicle.set("id", str(vehicle_id))
        vehicle_id += 1

    tree.write(routes_file)
    print("Updated:", routes_file)


print("Updating train vehicle IDs...")
update_vehicle_ids("train")
print("Updating test vehicle IDs...")
update_vehicle_ids("test")

Updating train vehicle IDs...
Updated: athens-osmWebWizard\train-routes.rou.xml
Updating test vehicle IDs...
Updated: athens-osmWebWizard\test-routes.rou.xml


## Vehicle Types


In [6]:
def update_vehicle_types(prefix):
    routes_file = os.path.join(DATASET_DIR, f"{prefix}-routes.rou.xml")
    if not os.path.exists(routes_file):
        print(f"Routes file not found: {routes_file}")
        return

    tree = ET.parse(routes_file)
    root = tree.getroot()

    for vehicle in root.findall("vehicle"):
        random_vehicle_type = np.random.choice(
            list(VEHICLE_TYPES_PROBABILITIES.keys()),
            p=list(VEHICLE_TYPES_PROBABILITIES.values()),
        )
        vehicle.set("type", random_vehicle_type)

    tree.write(routes_file)
    print("Updated:", routes_file)


print("Updating train vehicle types...")
update_vehicle_types("train")
print("Updating test vehicle types...")
update_vehicle_types("test")

Updating train vehicle types...
Updated: athens-osmWebWizard\train-routes.rou.xml
Updating test vehicle types...
Updated: athens-osmWebWizard\test-routes.rou.xml


## Fixed Routes


In [4]:
def generate_fixed_routes():
    fixed_flows_file = os.path.join(DATASET_DIR, "fixed-flows.rou.xml")
    fixed_routes_file = os.path.join(DATASET_DIR, "fixed-routes.rou.xml")
    fixed_routes_alt_file = os.path.join(DATASET_DIR, "fixed-routes.rou.alt.xml")

    command = [
        DUAROUTER_PATH,
        "--net-file",
        NETWORK_PATH,
        "--route-files",
        fixed_flows_file,
        "--output-file",
        fixed_routes_file,
    ]

    print("Executing:", " ".join(command))
    result = subprocess.run(command, capture_output=True, text=True)

    if result.stderr:
        print("Errors/Warnings from duarouter:")
        print(result.stderr)

    if os.path.exists(fixed_routes_file) and os.path.exists(fixed_routes_alt_file):
        os.remove(fixed_routes_alt_file)
        print("Success:", fixed_routes_file)
    else:
        print("Failed:", fixed_routes_file)


generate_fixed_routes()

Executing: C:\Users\george\miniconda3\envs\thesis\Lib\site-packages\sumo\bin\duarouter.exe --net-file athens-osmWebWizard\osm.net.xml.gz --route-files athens-osmWebWizard\fixed-flows.rou.xml --output-file athens-osmWebWizard\fixed-routes.rou.xml
Success: athens-osmWebWizard\fixed-routes.rou.xml
